# 전차와 일반 자동차 인식

주어진 폴더(`data/tank_car`)에 있는 이미지 파일들을 처리하여 각 파일별로 나타난 전차와 일반 자동차의 수를 인식하는 것이 이번 문제의 목표입니다.

YOLOv8 모델을 사용하는 것을 기본으로 합니다. 단 YOLOv8 모델은 tank를 인식하지 못하기에 tank가 일반적으로 어떤 객체 타입으로 인식되는지 확인해보고 그 특징을 이용하여 일반 자동차(car)와 다르게 인식하는 방법을 추천합니다.

## 입력 데이터의 내용

입력 폴더는 `data/tank_car`에 있으며, 이 안에는 `imageNNN` 형식으로 된 400개의 이미지 파일이 있습니다. 여기서 `NNN`은 1부터 시작하는 세 자리 숫자입니다. 즉 첫번째 이미지 파일의 이름은 `image001`이 됩니다. 이미지 파일의 파일 확장자가 다를 수 있습니다 (대부분 png, jpg, jpeg입니다). 파일 이름은 다르지만 동일한 내용을 갖는 중복 이미지들이 일부 존재할 수 있습니다. 참고로 일반 자동차와 전차가 동시에 한 이미지에 존재하는 경우는 없고 하나가 아닌 다수의 전차나 자동차가 한 이미지에 나타날 수 있습니다. 아래 이미지 2개를 예시로 보였습니다. 일반 자동차 혹은 전차가 일부 잘려 있어도 해당 객체의 수를 세야 합니다.

<img src='data/tank_car/image001.jpg'>
<img src='data/tank_car/image005.jpg'>

## 초기코드

아래 초기 코드를 모두 실행하여 패키지를 설치한 후 반드시 커널을 재시작해야 합니다. `Kernel → Restart`

In [ ]:
# 🧱 Step 1: YOLOv8 설치
!pip uninstall pandas numpy -y
!pip install ultralytics numpy==1.26.4 pandas==2.2.2

Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached pandas-2.2.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (19 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
Using cached pandas-2.2.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (12.7 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is

In [ ]:
# PyTorch 완전히 제거 후 재설치
!pip uninstall torch torchvision torchaudio -y
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

Found existing installation: torch 2.5.1+cu121
Uninstalling torch-2.5.1+cu121:
  Successfully uninstalled torch-2.5.1+cu121
Found existing installation: torchvision 0.20.1+cu121
Uninstalling torchvision-0.20.1+cu121:
  Successfully uninstalled torchvision-0.20.1+cu121
Found existing installation: torchaudio 2.5.1+cu121
Uninstalling torchaudio-2.5.1+cu121:
  Successfully uninstalled torchaudio-2.5.1+cu121
Looking in indexes: https://download.pytorch.org/whl/cpu
  Using cached https://download.pytorch.org/whl/cpu/torch-2.8.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached https://download.pytorch.org/whl/cpu/torchvision-0.23.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.1 kB)
  Using cached https://download.pytorch.org/whl/cpu/torchaudio-2.8.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (7.2 kB)
Using cached https://download.pytorch.org/whl/cpu/torch-2.8.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl (183.9 MB)
Using cached https://download

## 최종 제출 파일

위 폴더(`data/tank_car`) 내의 파일들을 하나씩 읽어서 이를 처리하고 인식된 일반 자동차와 전차의 수를 이미지 파일의 이름과 함께 아래와 같은 형태의 CSV 파일로 저장하고 파일의 이름은 `submission.csv`라고 지정합니다. 이때 결과는 이미지 번호(NNN)를 기준으로 오름차순 정렬되어야 합니다. 아래는 `submission.csv` 파일의 처음 5개 이미지 결과를 보여주는 예시입니다.(포맷을 보여주기 위한 예일 뿐입니다.)

| filename | car | tank |
| --- | --- | --- |
| image001.jpg | 1 | 0 |
| image002.jpg | 1 | 0 |
| image003.jpeg | 0 | 2 |
| image004.jpg | 0 | 1 |
| image005.jpg | 0 | 1 |

 - filename: 이미지 파일명
 - car: 해당 이미지에서 탐지된 일반 자동차 수
 - tank: 해당 이미지에서 탐지된 전차 수

## 평가 기준

제출한 결과는 정확도(Accuracy)를 기준으로 계산합니다. 즉, 이미지별로 car와 tank의 개수가 정확히 맞은 퍼센트를 점수로 사용합니다.

In [ ]:
import os, re, glob
import numpy as np
import pandas as pd
from PIL import Image
from ultralytics import YOLO

# ---------- 1. 경로 설정 (Colab/로컬 후보 중 존재하는 곳 사용) ----------
CANDIDATES = [
    'data/tank_car', '/content/data/tank_car', '/content/drive/MyDrive/대회/data1/tank_car',
]
DATA_DIR = next(p for p in CANDIDATES if os.path.isdir(p))
print('DATA_DIR =', DATA_DIR)

# ---------- 2. 모델 ----------
# detect : COCO 사전학습 YOLOv8s (tank 클래스 없음 -> 전차는 truck/train/boat 등으로 검출됨)
# cls    : ImageNet 사전학습 YOLOv8m-cls (ImageNet에 'tank' 클래스가 있어 이미지 단위 car/tank 판별에 사용)
det = YOLO('yolov8s.pt')
cls = YOLO('yolov8m-cls.pt')

TANKISH = {'tank', 'half_track', 'tracked_vehicle', 'assault_rifle', 'military_uniform', 'cannon', 'projectile',
           'missile', 'bulldozer', 'harvester', 'tractor', 'crane', 'snowplow'}
CARSET = {'sports_car', 'beach_wagon', 'minivan', 'grille', 'car_wheel', 'limousine', 'convertible', 'racer',
          'cab', 'pickup', 'jeep', 'Model_T', 'police_van', 'minibus'}
CAR_CLASSES = {'car', 'truck', 'bus'}
# 전차는 COCO에서 여러 클래스로 오검출되므로 차량 계열 + 오검출 후보 클래스를 모두 후보로 사용(person 제외)
TANK_CLASSES = {'car', 'truck', 'bus', 'train', 'boat', 'airplane', 'motorcycle', 'bicycle',
                'suitcase', 'cake', 'potted plant', 'dog', 'clock'}

def area(b): return (b[2] - b[0]) * (b[3] - b[1])
def inter_over_min(a, b):
    w = max(0, min(a[2], b[2]) - max(a[0], b[0])); h = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    return w * h / (min(area(a), area(b)) + 1e-9)

def count_objects(boxes, allowed, merge_thr=0.6, min_rel=0.05):
    # 큰 박스부터 보며, 이미 채택된 박스에 60% 이상 포함되는 박스는 중복으로 제거,
    # 가장 큰 박스의 5% 미만 면적인 박스는 부품/오검출로 제거
    cand = sorted([b for c, b in boxes if c in allowed], key=area, reverse=True)
    keep = []
    for b in cand:
        if all(inter_over_min(b, k) < merge_thr for k in keep):
            keep.append(b)
    if keep:
        keep = [k for k in keep if area(k) >= min_rel * area(keep[0])]
    return len(keep)

def predict(path):
    # 읽을 수 없는 파일(0바이트 등)은 검출할 대상이 없으므로 car=0, tank=0 으로 처리
    try:
        im = Image.open(path).convert('RGB')
    except Exception as e:
        print('unreadable ->', os.path.basename(path), e)
        return 0, 0
    # (1) 이미지 단위 판별: tank / car
    p = cls(im, verbose=False)[0].probs
    names = cls.names
    tank_score = float(sum(p.data[i] for i, n in names.items() if n in TANKISH))
    top1 = names[int(p.top1)]
    is_tank = tank_score > 0.1 or top1 not in CARSET
    # (2) 객체 수 계산
    r = det(im, verbose=False, conf=0.25)[0]
    boxes = [(det.names[int(c)], b) for c, b in zip(r.boxes.cls.tolist(), r.boxes.xyxy.tolist())]
    if is_tank:
        return 0, max(1, count_objects(boxes, TANK_CLASSES))   # 전차 이미지는 최소 1대
    return count_objects(boxes, CAR_CLASSES), 0

# ---------- 3. 전체 이미지 추론 (이미지 번호 오름차순) ----------
files = [f for f in os.listdir(DATA_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))]
files.sort(key=lambda f: int(re.search(r'image(\d+)', f).group(1)))
rows = []
for f in files:
    car, tank = predict(os.path.join(DATA_DIR, f))
    rows.append({'filename': f, 'car': car, 'tank': tank})
submission_df = pd.DataFrame(rows)
print(len(submission_df), 'images |', 'car imgs:', int((submission_df.car > 0).sum()), '/ tank imgs:', int((submission_df.tank > 0).sum()))
submission_df.head()

## 결과 저장

채점을 위해, 위에서 구한 데이터를 현재 파일과 같은 디렉토리(.ipynb 파일이 있는 디렉토리)에 `submission.csv`이라는 이름으로 저장해 주세요.

In [ ]:
# 제출 파일 저장 (index 열 없음) + 검증
submission_df.to_csv('submission.csv', index=False)
chk = pd.read_csv('submission.csv')
assert list(chk.columns) == ['filename', 'car', 'tank'] and len(chk) == len(files)
assert chk.filename.is_unique and (chk[['car', 'tank']] >= 0).all().all()
assert ((chk.car > 0) & (chk.tank > 0)).sum() == 0     # 문제 조건: car와 tank는 한 이미지에 공존하지 않음
chk.head()